importation des libs et chargement des datas

In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px  # Interface haut niveau pour graphiques simples
import plotly.graph_objects as go  # Interface bas niveau pour contrôle précis
import seaborn as sns
from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots  # Création de grilles de graphiques

#Affichage complet des colonnes et des lignes (pas de retour a la ligne)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.expand_frame_repr", False)

#Import du fichier csv customer
url = "./../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url,sep=",")

# Affichage des dimension du jeu de données
print(f"Le jeu de données a {df.shape[0]} lignes et {df.shape[1]} colonnes")

# Affichez les 5 premières lignes
print(df.head())




Le jeu de données a 7043 lignes et 21 colonnes
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  MonthlyCharges TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes           Electronic check           29.85        29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No               Mailed check           56.95       1889.5    No
2  3668-QPYBK    Male              0      No         No       2          Yes    

Affichez les types de données de chaque colonne

Nombre pour chaque variable

In [4]:
# Affichez les types de données de chaque colonne
print(df.dtypes)
print(" " * 50)
print("-" * 50)
print(" " * 50)
#Nombre de manquants pour chaque colonnes

# remplace les chaînes vides par NaN dans les colonnes de type objet ou string
cols_str = df.select_dtypes(include=["object", "string"]).columns
df[cols_str] = df[cols_str].replace(r"^\s*$", np.nan, regex=True)


if (df.isna().sum() == 0).all():
    print("Il ne manque aucune valeur.")
else:
    print("Il y a des valeurs manquantes.")
    print(f"{df.isna().sum()}")  

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
                                                  
--------------------------------------------------
                                                  
Il y a des valeurs manquantes.
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineS

Recherche de doublon

In [5]:
#Recherhce de doublon
print(f"Il y a {df.duplicated().sum()} doublons dans le jeu de données.")
if df.duplicated().sum() > 0:
    print("Voici les lignes en double :")
    print(df[df.duplicated()])

Il y a 0 doublons dans le jeu de données.


Recherche des valeur aberrantes (outlier)
1 Visualisation (boxpplot) sur les valeurs numerique
2 tableau recaptulatif des valeures yes ou no pour les colonnes "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"
        idem pour MultipleLines mais il y a 3 valeurs possible (Yes/No/No phone service)
    si besoin 2 approche
        1 Stat
        2 Machine Learning

In [6]:

# Creaton DataFrame pour ne garder que les valeur numerique pour le boxplot (SeniorCitizen, tenure, MonthlyCharges et un global pour les trois)
df_numeric = df.select_dtypes(include=[np.number])
df_SeniorCitizen = df_numeric["SeniorCitizen"]
df_tenure = df_numeric["tenure"]
df_MonthlyCharges = df_numeric["MonthlyCharges"]
#Creation d'une figure 1 ligne 3 colonnes
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Distribution DSeniorCitizen', 'Distribution Dtenure',' Distribution MonthlyCharges')
)

fig.add_trace(go.Box(y=df_SeniorCitizen, name="SeniorCitizen"), row=1, col=1)
fig.update_xaxes(title_text="SeniorCitizen", row=1, col=1)
fig.update_yaxes(title_text="ancienneté 1/0", row=1, col=1)

fig.add_trace(go.Box(y=df_tenure, name="tenure"), row=1, col=2)
fig.update_xaxes(title_text="tenure", row=1, col=2)
fig.update_yaxes(title_text="Ancienneté client (mois)", row=1, col=2)

fig.add_trace(go.Box(y=df_MonthlyCharges, name="MonthlyCharges"), row=1, col=3)
fig.update_xaxes(title_text="MonthlyCharges", row=1, col=3)
fig.update_yaxes(title_text="Facturation mensuelle", row=1, col=3)


# Affichage du graphique
fig.show()

pas de valeur aberrante

Recherche de incoherances

In [7]:
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())

In [8]:
#Recherche valeur autre que Yes No pour "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"

cols_yes_no = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn","MultipleLines"]
valeurs_autorisees = ["Yes", "No","No phone service"]

for col in cols_yes_no:
    s = df[col]
    #selection des valeurs non NaN qui ne sont pas dans les valeurs autorisées
    if col == "MultipleLines":
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_autorisees)]
    else:
        #pour les colonnes autre que MultipleLines, les valeurs autorisées sont uniquement "Yes" et "No"
        invalides = s[s.notna() & ~s.astype(str).str.strip().isin(valeurs_autorisees[:2])]
        

    
    if invalides.empty:
        print(f"{col}: OK (NaN={s.isna().sum()})")
    else:
        print(f"{col}: INVALIDES -> {invalides.unique()} (NaN={s.isna().sum()})")



Partner: OK (NaN=0)
Dependents: OK (NaN=0)
PhoneService: OK (NaN=0)
PaperlessBilling: OK (NaN=0)
Churn: OK (NaN=0)
MultipleLines: OK (NaN=0)


Recap du controle qualité

In [9]:
# 1) Valeurs manquantes
missing_counts = df.isna().sum()
missing_total = int(missing_counts.sum())

# 2) Doublons (lignes complètes)
duplicates_count = int(df.duplicated().sum())

# 3) Cohérences métier
inco_tenure = int((df["tenure"] < 0).sum())
inco_monthly = int((df["MonthlyCharges"] < 0).sum())


# 4) Résumé
print("=== Résumé qualité des données ===")
print(f"Valeurs manquantes totales : {missing_total}")
print(f"Doublons (lignes complètes) : {duplicates_count}")
print(f"Incohérences tenure < 0 : {inco_tenure}")
print(f"Incohérences MonthlyCharges < 0 : {inco_monthly}")


print("\nDétail valeurs manquantes (colonnes avec au moins 1 NaN) :")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

if (
    missing_total == 0
    and duplicates_count == 0
    and inco_tenure == 0
    and inco_monthly == 0
       
    ):
    print("\nConclusion : controles qualite OK.")
else:
    print("\nConclusion : il reste des points a traiter.")

=== Résumé qualité des données ===
Valeurs manquantes totales : 11
Doublons (lignes complètes) : 0
Incohérences tenure < 0 : 0
Incohérences MonthlyCharges < 0 : 0

Détail valeurs manquantes (colonnes avec au moins 1 NaN) :
TotalCharges    11
dtype: int64

Conclusion : il reste des points a traiter.


Exploration de la cible:
    taux Global de churn
    puis par segments 
        Contrat
        InternetService
        tenure
        Payment methode
        gender
        Partner
        Dependents
        PhoneService
        MultipleLines
        InternetService
        OnlineSecurity
        OnlineBackup
        DeviceProtection
        TechSupport
        StreamingTV
        StreamingMovies
        PaperlessBilling
        PaymentMethod
        MonthlyCharges

In [10]:
#Taux global de churn
taux_global_churn = (df["Churn"] == "Yes").mean()
print(f"Taux global de churn (Pourcentage de personne qui ont résilié) : {taux_global_churn:.2%}")

Taux global de churn (Pourcentage de personne qui ont résilié) : 26.54%


Definition du protocole d'evalutaion

Split : train / validation / test (stratifié  Churn)
Métriques : ROC-AUC, Precision, Recall, F1-score
Déséquilibre : prise en compte via l’analyse de ces métriques (notamment Recall/F1 sur churn)

In [27]:


# =========================
# 1) Préparation des données
# =========================

# Copie de sécurité (optionnel)
df_work = df.copy()

# Normaliser Churn en 0/1
df_work["Churn_num"] = (
    df_work["Churn"]
    .astype(str).str.strip().str.lower()
    .map({
        "yes": 1, "no": 0,
        "1": 1, "0": 0,
        "true": 1, "false": 0
    })
)

# Vérification rapide (valeurs non reconnues)
n_na = df_work["Churn_num"].isna().sum()
print(f"Valeurs Churn non reconnues: {n_na}")

# Segment tenure (si la colonne tenure existe)
if "tenure" in df_work.columns:
    bins = [0, 12, 24, 36, 48, 60, 72]
    labels = ["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"]
    df_work["tenure_segment"] = pd.cut(
        df_work["tenure"], bins=bins, labels=labels, include_lowest=True
    )

# =========================
# 2) Fonction unique bar chart
# =========================

def plot_churn_rate_bar(data, segment_col, title=None):
    tmp = (
        data.groupby(segment_col, dropna=False)["Churn_num"]
            .mean()
            .mul(100)
            .reset_index(name="churn_rate")
            .sort_values("churn_rate", ascending=False)
    )

    print(f"\n--- {segment_col} ---")
    print(tmp)

    fig = px.bar(
        tmp,
        x=segment_col,
        y="churn_rate",
        color=segment_col,  # une couleur différente par barre
        text=tmp["churn_rate"].round(1).astype(str) + "%",
        labels={segment_col: segment_col, "churn_rate": "Taux de churn (%)"},
        title=title or f"Taux de churn par {segment_col}"
    )

    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        xaxis_tickangle=-25,
        yaxis_range=[0, max(5, tmp["churn_rate"].max() * 1.15)]
    )
    fig.show()

# =========================
# 3) Générer tous les graphes
# =========================

segments = [
    "tenure_segment",   # créé plus haut
    "Contract",
    "PaymentMethod",
    "PhoneService",
    "OnlineSecurity",
    "MultipleLines"
]

for col in segments:
    if col in df_work.columns:
        plot_churn_rate_bar(df_work, col, f"Taux de churn par {col}")
    else:
        print(f"Colonne absente: {col}")


Valeurs Churn non reconnues: 0

--- tenure_segment ---
  tenure_segment  churn_rate
0           0-12   47.438243
1          13-24   28.710938
2          25-36   21.634615
3          37-48   19.028871
4          49-60   14.423077
5          61-72    6.609808



--- Contract ---
         Contract  churn_rate
0  Month-to-month   42.709677
1        One year   11.269518
2        Two year    2.831858



--- PaymentMethod ---
               PaymentMethod  churn_rate
2           Electronic check   45.285412
3               Mailed check   19.106700
0  Bank transfer (automatic)   16.709845
1    Credit card (automatic)   15.243101



--- PhoneService ---
  PhoneService  churn_rate
1          Yes   26.709637
0           No   24.926686



--- OnlineSecurity ---
        OnlineSecurity  churn_rate
0                   No   41.766724
2                  Yes   14.611194
1  No internet service    7.404980



--- MultipleLines ---
      MultipleLines  churn_rate
2               Yes   28.609896
0                No   25.044248
1  No phone service   24.926686
